In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Downloaded to:", path)

In [ ]:
import os

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
img_dir_candidates = [root for root, dirs, files in os.walk(path) if any(f.lower().endswith(".jpg") for f in files)]

print("CSV files found:", csv_candidates)
print("Image directories found:", img_dir_candidates)

In [ ]:
import pandas as pd

metadata_path = next(p for p in csv_candidates if "metadata" in p.lower())
df = pd.read_csv(metadata_path)
print(df.shape)
print(df.columns.tolist())

class_counts = df["dx"].value_counts()
print()
print(class_counts)
print()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio (largest/smallest class): {imbalance_ratio:.1f}x")
majority_baseline_acc = class_counts.max() / len(df)
print(f"'Always predict {class_counts.idxmax()}' baseline accuracy: {majority_baseline_acc:.1%}")

In [ ]:
def find_image_path(image_id):
    for img_dir in img_dir_candidates:
        candidate = os.path.join(img_dir, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None

df["image_path"] = df["image_id"].apply(find_image_path)

missing = df["image_path"].isna().sum()
print(f"Images not found: {missing} out of {len(df)}")
df = df.dropna(subset=["image_path"]).reset_index(drop=True)
print("Final shape:", df.shape)

In [ ]:
import numpy as np

DX_TO_FULLNAME = {
    "akiec": "Actinic keratoses and intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}
DX_CODES = sorted(DX_TO_FULLNAME.keys())  # fixed order — this defines the model's output index order
CONDITIONS = [DX_TO_FULLNAME[c] for c in DX_CODES]

code_to_index = {code: i for i, code in enumerate(DX_CODES)}
df["label"] = df["dx"].map(code_to_index)

print("Class index mapping:")
for code, idx in code_to_index.items():
    print(f"  {idx}: {code} -> {DX_TO_FULLNAME[code]}")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
import tensorflow as tf

IMG_SIZE = 128

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image, label

def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df["image_path"].to_numpy(), train_df["label"].to_numpy(), shuffle=True)
val_ds = make_dataset(val_df["image_path"].to_numpy(), val_df["label"].to_numpy())

for imgs, labs in train_ds.take(1):
    print("batch image shape:", imgs.shape, "batch label shape:", labs.shape)

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(CONDITIONS), activation="softmax"),  # softmax, not sigmoid — mutually exclusive classes
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

callbacks = [EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    shuffle=False,
    callbacks=callbacks,
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = val_df["label"].to_numpy()
y_pred_probs = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = (y_pred == y_true).mean()
print(f"Model accuracy: {accuracy:.1%}")
print(f"'Always predict majority class' baseline: {majority_baseline_acc:.1%}")
print(f"Difference: {(accuracy - majority_baseline_acc)*100:+.1f} points")
print()
print(classification_report(y_true, y_pred, target_names=CONDITIONS, zero_division=0))
print()
print("Confusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(y_true, y_pred))

In [ ]:
import json, os

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v1_baseline_cnn.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

In [ ]:
import shutil
shutil.make_archive("v1_skin_lesion_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v1_skin_lesion_artifacts.zip")